# 📓 Semana 15 · Dia 1 — Arquiteturas: ReAct, Reflexion e Multi-agente

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | GenAI Engineer Associate |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Multi-agente com coordenador rodando |

---


## 📖 Teoria — As 3 arquiteturas

| Arquitetura | Ideia | Quando |
|---|---|---|
| **ReAct** | raciocina → age → observa | tarefas com tools |
| **Reflexion** | gera → **avalia** → corrige | qualidade crítica (código, SQL) |
| **Multi-agente** | coordenador delega a especialistas | tarefas compostas |

Reflexion = ReAct + feedback: o agente critica a própria resposta e refaz até passar nos critérios.


### 💻 Na prática — Reflexion (gera → avalia → corrige)

Implemente o loop de auto-melhoria.


In [ ]:
# Reflexion para SQL: gera, valida, corrige até 3x
def reflexion_sql(pergunta, max_rounds=3):
    criticas = ""
    for ronda in range(max_rounds):
        prompt = (dicionario + "\nPergunta: " + pergunta
                  + "\nCríticas anteriores: " + criticas
                  + "\nGere o SQL:")
        sql_gerado = llm.invoke(prompt).content.strip().strip("```")
        try:
            valida_sql(sql_gerado)
            df = spark.sql(sql_gerado)
            return sql_gerado, df.toPandas()
        except Exception as e:
            criticas += f"\n- Falhou ({e}). Corrija o SQL."
            print(f"Ronda {ronda+1}: corrigindo ({e})")
    return None, "Não consegui gerar SQL válido."
print("Reflexion implementado (gera → crítica → corrige).")

### 💻 Na prática — Multi-agente

Coordenador delega para agentes especialistas.


In [ ]:
# Especialistas (cada um com sua tool)
from langgraph.graph import StateGraph, END
from typing import TypedDict
class Estado(TypedDict):
    pergunta: str
    resposta: str
    especialista: str

def coordenador(estado):
    p = estado["pergunta"].lower()
    if "receita" in p or "venda" in p:
        estado["especialista"] = "vendas"
    elif "produto" in p or "estoque" in p:
        estado["especialista"] = "produtos"
    else:
        estado["especialista"] = "geral"
    return estado

def agente_vendas(estado):
    estado["resposta"] = agente_final.invoke({"input": estado["pergunta"]})["output"]
    return estado

def agente_produtos(estado):
    estado["resposta"] = top_produtos(3)
    return estado

def agente_geral(estado):
    estado["resposta"] = "Assistente geral: " + estado["pergunta"]
    return estado
print("Especialistas definidos.")

In [ ]:
# Grafo multi-agente
g = StateGraph(Estado)
g.add_node("coordenador", coordenador)
g.add_node("vendas", agente_vendas)
g.add_node("produtos", agente_produtos)
g.add_node("geral", agente_geral)
g.set_entry_point("coordenador")
g.add_conditional_edges("coordenador",
    lambda e: e["especialista"])
for n in ["vendas", "produtos", "geral"]:
    g.add_edge(n, END)
multi = g.compile()
print(multi.invoke({"pergunta": "Qual a receita de novembro?"})["resposta"][:100])

> 🎯 **Dica de prova**: Agentes: Reflexion = avaliar e corrigir; Multi-agente = delegar. Pergunta: 'como melhorar a precisão de um agente de SQL?' → Reflexion com crítica.


## 🎯 Exercícios de fixação

**1.** Adicione um especialista 'graficos' ao multi-agente.

**2.** O que o Reflexion adiciona ao ReAct?

**3.** Quando NÃO usar multi-agente?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Especialista

Crie nó que gera código de gráfico (matplotlib) e rota no coordenador.

**2.** Reflexion

Um passo de avaliação/crítica entre geração e entrega — qualidade sobre velocidade.

**3.** Sem multi-agente

Tarefas simples: multi-agente adiciona latência e complexidade — use um agente único.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*